<a href="https://colab.research.google.com/github/lindajia9/IDXExchange-ds58/blob/main/04_model_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_absolute_percentage_error
)

## Load Test and Training Sets

In [33]:
train_df = pd.read_csv("train_preprocessed2.csv")

test_df = pd.read_csv("test_preprocessed2.csv")

/tmp/ipykernel_495/1321383357.py:1: DtypeWarning: Columns (43,51) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv("train_preprocessed2.csv")
/tmp/ipykernel_495/1321383357.py:3: DtypeWarning: Columns (43) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df = pd.read_csv("test_preprocessed2.csv")


In [34]:
city_cols = [col for col in train_df.columns if col.startswith("City_")]
county_cols = [col for col in train_df.columns if col.startswith("CountyOrParish_")]

features = [
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeArea",
    "Age",
    "Latitude",
    "Longitude",
] + city_cols + county_cols

In [35]:
# Separate predictors and target variable for training and testing sets

X_train = train_df[features]
y_train = train_df['ClosePrice']
y_train_log = np.log1p(y_train)

X_test = test_df[features]
y_test = test_df['ClosePrice']


## Function to calculate metrics:

In [36]:
def evaluate_model(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)

    mae = mean_absolute_error(y_true, y_pred)

    mape = mean_absolute_percentage_error(y_true, y_pred) * 100

    ape = np.abs((y_true - y_pred) / y_true) * 100
    mdape = np.median(ape)

    return r2, mae, mape, mdape

## Fit Models

In [37]:
# Baseline: Linear Regression

lr_model = LinearRegression()

lr_model.fit(X_train, y_train_log)

lr_pred_log = lr_model.predict(X_test)

# Convert back to dollars
lr_pred = np.expm1(lr_pred_log)


# Decision Tree Regressor
dt_model = DecisionTreeRegressor(
    max_depth=20,
    min_samples_leaf=5,
    random_state=42
)

dt_model.fit(X_train, y_train_log)

dt_pred_log = dt_model.predict(X_test)

dt_pred = np.expm1(dt_pred_log)


In [38]:
# Random Forest Regressor
rf_model = RandomForestRegressor(
    n_estimators=50,
    max_depth=20,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train_log)

rf_pred_log = rf_model.predict(X_test)

# Convert back to dollars
rf_pred = np.expm1(rf_pred_log)

In [39]:
# Linear Regression
lr_metrics = evaluate_model(y_test, lr_pred)

# Decision Tree
dt_metrics = evaluate_model(y_test, dt_pred)

# Random Forest
rf_metrics = evaluate_model(y_test, rf_pred)

## Put results into a table

In [40]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "R²": [
        lr_metrics[0],
        dt_metrics[0],
        rf_metrics[0]
    ],
    "MAE": [
        lr_metrics[1],
        dt_metrics[1],
        rf_metrics[1]
    ],
    "MAPE (%)": [
        lr_metrics[2],
        dt_metrics[2],
        rf_metrics[2]
    ],
    "MdAPE (%)": [
        lr_metrics[3],
        dt_metrics[3],
        rf_metrics[3]
    ]
})

results

,Model,R²,MAE,MAPE (%),MdAPE (%)
0,Linear Regression,-34.897430,374562.203165,21.866775,11.577040
1,Decision Tree,0.474360,261845.652813,18.816081,11.089356
2,Random Forest,0.538683,217320.454654,16.907601,9.092034


## Comparing Results

After applying a logarithmic transformation to ClosePrice, the Random Forest Regressor achieved the highest performance with an R² of 0.539 and the lowest MAE, MAPE, and MdApe. The improvement is likely due to reducing the influence of extreme price outliers and allowing the models to better capture nonlinear relationships between property characteristics and price.




## Document model behavior
**Linear Regression**

Strengths:



*   Simple and easy to interpret
*   Estimates the relationship between features and price using coefficients

*   Works well when relationships are approximately linear



Weaknesses:

*  Cannot naturally capture complex nonlinear relationships

*   Sensitive to outliers

*   Assumes each feature has a linear effect on price.

**Decision Tree Regressor**

Strengths:

*   Captures nonlinear relationships automatically
*   Can model interactions between variables (for example, location + living area)


*   Does not require feature scaling




Weaknesses:


*  Can overfit easily by creating very specific rules

*   A small change in data can produce a very different tree
*   Usually performs worse than ensembles like Random Forest


**Random Forest Regressor**

Strengths:



*   Combines many decision trees to reduce overfitting

*   Captures nonlinear relationships and feature interactions
*  Usually gives strong predictive performance on tabular data


Weaknesses:

*  Less interpretable than Linear Regression or one Decision Tree

*   Requires more computation
*   Can still overfit if not tuned